# Run 1307 Geothermal AI in Colab (Git code + GCS data)

This notebook is meant to be opened from GitHub in Google Colab.
By default it **clones/updates the GeothermalAI repo** and runs scripts from `Git_1307/`,
while **large rasters and run outputs** still use **GCS** (`GCS_EXTRA_SYNC_PREFIXES`, outputs under `GCS_OUTPUT_PREFIX`).
Set `USE_GIT_FOR_CODE = False` to use the older flow that **rsyncs 1307 code from GCS** into `/content/1307` instead.
It can **upload the tile dataset as a single `.tar.gz` to GCS** so work survives Colab disconnects without slow per-file sync.

### Your Windows PC: `gcloud` and `gsutil` (optional)

Use this when you run **`gsutil`** from **PowerShell** on your laptop (for example syncing data to your bucket outside of Colab).

1. Install the [Google Cloud CLI](https://cloud.google.com/sdk/docs/install) — it includes **both** `gcloud` and `gsutil`.
2. **Restart Cursor** or open a **new** terminal so **PATH** includes `...\google-cloud-sdk\bin` (otherwise `gcloud` may be “not recognized”).
3. Confirm tools work:
   - `gcloud --version`
   - `gsutil version`
4. When you use this project on a machine for the first time (or after a long break):
   - `gcloud auth login`
   - `gcloud config set project YOUR_PROJECT_ID`  
   Use the **Project ID** from [Google Cloud Console](https://console.cloud.google.com/) (lowercase id), not the display name.

**Colab** uses the notebook’s in-runtime Google sign-in — that is **separate** from your PC. Keep **`GCP_PROJECT`** in the config cell below aligned with the same project.

## Progress tracker (edit as you go)

- [ ] Section 2: Git clone (or GCS 1307 sync) + data prefixes from GCS
- [ ] Section 4 dependencies installed
- [ ] Section 5 GPU confirmed (`nvidia-smi` works)
- [ ] Section 6 run directory created
- [ ] Before Section 7 local inputs prepared
- [ ] Section 7 built dataset or set `DOE_DATASET_PATH`
- [ ] Section 8 `gsutil` sync of the run folder to GCS

### Next steps
- Set or verify `DOE_GRI_INPUT`.
- Build dataset (or set `DOE_DATASET_PATH` to an existing dataset directory).
- Run training.
- Sync run outputs to GCS; dataset is uploaded as one `.tar.gz` when `SYNC_DATASET_TO_GCS` is on.

## 1) Configure

Fill in these values before running the rest of the notebook.

In [ ]:
GCP_PROJECT = "maloney-geog-473"
GCS_BUCKET = "gis-final-project"
# When USE_GIT_FOR_CODE is False, 1307 Python code is rsynced from this prefix into LOCAL_1307_DIR.
GCS_PREFIX_1307 = "GIS Final Project/1307"

USE_GIT_FOR_CODE = True  # True: clone GeothermalAI from GitHub and use Git_1307. False: rsync 1307 from GCS only.
GIT_REPO_URL = "https://github.com/GarretMaloney/GeothermalAI.git"  # must contain Git_1307/ (edit if you forked)
GIT_BRANCH = "main"
LOCAL_REPO_DIR = "/content/GeothermalAI"  # clone target; repo root (not the Git_1307 subfolder)
CODE_SUBDIR = "Git_1307"  # folder under LOCAL_REPO_DIR with doe_geoai.py, create_doe_dataset.py, doe_tiff
LOCAL_1307_DIR = f"{LOCAL_REPO_DIR.rstrip('/')}/{CODE_SUBDIR}"

# Data on GCS (rasters, etc.) — not the Python training code when USE_GIT_FOR_CODE is True.
GCS_EXTRA_SYNC_PREFIXES = [
    "GIS Final Project/BradyGDB/BradyRaw/Brady_Analysis/Geophysics/BradySOM"
]

REQUIREMENTS_FILE = "requirements.txt"  # if missing under Git_1307, section 4 falls back to repo root requirements.txt
EXTRA_PIP_PACKAGES = ""  # optional, space-separated

# Default flow: train from the saved Brady tile dataset in GCS.
# Switch ENTRY_SCRIPT to "create_doe_dataset.py" only when you need to rebuild tiles.
ENTRY_SCRIPT = "doe_geoai.py"
ENTRY_ARGS = ""  # blank: auto args from DOE_* (dataset build) or training knobs (doe_geoai)

DOE_GRI_INPUT = "/content/GIS Final Project/BradyGDB/BradyRaw/Brady_Analysis/Geophysics/BradySOM/brady_som_output.gri"
DOE_DATASET_OUT_DIR = "/content/doe-data/brady_samples_19x3d"
DOE_CHANNELS = 3
DOE_SAMPLE_COUNT = 100000
DOE_KERNEL_PIXELS = 19

DOE_DATASET_PATH = "/content/doe-data/brady_samples_19x3d"  # local folder after extracting GCS_DATASET_ARCHIVE
AUTO_DOWNLOAD_DATASET_FROM_GCS = True  # for doe_geoai: pull/extract the saved tile .tar.gz if DOE_DATASET_PATH is missing
GCS_DATASET_ARCHIVE = "GIS Final Project/outputs/doe-datasets/brady_samples_19x3d.tar.gz"
DOE_LABELBIN_PATH = ""
DOE_MODEL_PATH = ""
DOE_PLOT_PATH = ""
DOE_CURVES_PATH = ""

# Current short run settings. Moraga et al. report 100 epochs, 19x19x3 tiles, rotation/mirror augmentation,
# Brady split about 6.7% train / 6.7% validation / 20.2% test; Desert Peak about 1.6% / 1.6% / 4.7%.
DOE_EPOCHS = 25
DOE_BATCH_SIZE = 32
DOE_GPUS = 1
DOE_EXTRA_ARGS = ""

# Persistent run outputs in GCS.
GCS_OUTPUT_PREFIX = "GIS Final Project/outputs/1307"
SYNC_DATASET_TO_GCS = True
GCS_DATASET_PREFIX = "GIS Final Project/outputs/doe-datasets"  # under bucket; a single .tar.gz is uploaded (not per-.npy rsync)
RUN_NAME_OVERRIDE = ""  # blank = timestamp for /content/1307_runs/...

# Optional: auto-append output args for scripts that support these flags.
AUTO_APPEND_OUTPUT_ARGS = False
OUTPUT_DIR_FLAG = "--output_dir"
SAVE_DIR_FLAG = "--save_dir"

## 2) Authenticate; clone/update code from Git (or sync 1307 from GCS); sync data from GCS

In [ ]:
import shlex
import subprocess
from pathlib import Path

from google.colab import auth

def run(cmd, cwd=None):
    print("$", " ".join(shlex.quote(str(c)) for c in cmd))
    p = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.returncode != 0:
        if p.stderr:
            print(p.stderr)
        raise subprocess.CalledProcessError(p.returncode, cmd, output=p.stdout, stderr=p.stderr)
    return p

if not GCP_PROJECT or not GCS_BUCKET:
    raise ValueError("Set GCP_PROJECT and GCS_BUCKET in the config cell first.")

auth.authenticate_user()
run(["gcloud", "config", "set", "project", GCP_PROJECT])

if USE_GIT_FOR_CODE:
    if not GIT_REPO_URL or not str(GIT_REPO_URL).strip():
        raise ValueError("Set GIT_REPO_URL when USE_GIT_FOR_CODE is True.")
    repo_path = Path(LOCAL_REPO_DIR)
    git_dir = repo_path / ".git"
    if git_dir.is_dir():
        run(["git", "-C", str(repo_path), "fetch", "origin", GIT_BRANCH])
        run(["git", "-C", str(repo_path), "checkout", GIT_BRANCH])
        run(["git", "-C", str(repo_path), "pull", "--ff-only", "origin", GIT_BRANCH])
        print("Updated git repo at", repo_path)
    else:
        if repo_path.exists() and any(repo_path.iterdir()):
            raise RuntimeError(
                f"{repo_path} exists but is not a git clone. Remove it in Colab (e.g. rm -r) or change LOCAL_REPO_DIR."
            )
        repo_path.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO_URL.strip(), str(repo_path)])
        print("Cloned", GIT_REPO_URL, "->", repo_path)
    code_dir = Path(LOCAL_1307_DIR)
    if not code_dir.is_dir():
        raise FileNotFoundError(f"After git sync, code dir is missing: {code_dir} (check CODE_SUBDIR in config).")
    print("Using training code from", code_dir)
else:
    local_dir = Path(LOCAL_1307_DIR)
    local_dir.mkdir(parents=True, exist_ok=True)
    _prefix = GCS_PREFIX_1307.strip("/")
    src = f"gs://{GCS_BUCKET}/{_prefix}" if _prefix else f"gs://{GCS_BUCKET}"
    run(["gsutil", "-m", "rsync", "-r", src, str(local_dir)])
    print("Synced 1307 code from GCS to", local_dir)

# Optional extra data syncs go under /content preserving relative path.
for extra_prefix in GCS_EXTRA_SYNC_PREFIXES:
    ep = extra_prefix.strip("/")
    if not ep:
        continue
    extra_src = f"gs://{GCS_BUCKET}/{ep}"
    extra_dst = Path("/content") / ep
    extra_dst.mkdir(parents=True, exist_ok=True)
    try:
        run(["gsutil", "-m", "rsync", "-r", extra_src, str(extra_dst)])
        print("Synced extra prefix via rsync:", extra_src, "->", extra_dst)
    except subprocess.CalledProcessError as exc:
        print("rsync failed for", extra_src)
        if exc.stderr:
            print(exc.stderr)
        print("Falling back to gsutil cp -r for this prefix...")
        try:
            run(["gsutil", "-m", "cp", "-r", extra_src, str(extra_dst.parent)])
            print("Synced extra prefix via cp -r:", extra_src, "->", extra_dst.parent)
        except subprocess.CalledProcessError as exc2:
            if exc2.stderr:
                print(exc2.stderr)
            raise RuntimeError(
                f"Failed syncing {extra_src}. "
                "Set GCS_EXTRA_SYNC_PREFIXES to exact small prefixes and rerun this cell."
            ) from exc2

## 3) Inspect local code files (`Git_1307` or GCS 1307 sync)

In [ ]:
from pathlib import Path

root = Path(LOCAL_1307_DIR)
if not root.exists():
    raise FileNotFoundError(f"Missing local code directory: {root}")

items = sorted(p.name for p in root.iterdir())
print(f"Top-level files/folders in {root}:")
for name in items[:200]:
    print(" -", name)

## 4) Install dependencies

In [ ]:
import sys
import shlex
from pathlib import Path

req_path = Path(LOCAL_1307_DIR) / REQUIREMENTS_FILE
req_repo = Path(LOCAL_REPO_DIR) / "requirements.txt" if USE_GIT_FOR_CODE else None

run([sys.executable, "-m", "pip", "install", "-U", "pip"])
if req_path.exists():
    run([sys.executable, "-m", "pip", "install", "-r", str(req_path)])
elif req_repo is not None and req_repo.is_file():
    print("Using repo root requirements:", req_repo)
    run([sys.executable, "-m", "pip", "install", "-r", str(req_repo)])
else:
    print(f"No requirements file at {req_path}" + (f" or {req_repo}" if req_repo else ""))

extra = EXTRA_PIP_PACKAGES.strip()
if extra:
    run([sys.executable, "-m", "pip", "install", *shlex.split(extra)])

## 5) Check GPU runtime

In [ ]:
import subprocess

try:
    run(["nvidia-smi"])
except (subprocess.CalledProcessError, FileNotFoundError, OSError):
    print("nvidia-smi not available. In Colab: Runtime -> Change runtime type -> GPU if you need a GPU.")

try:
    import torch
    print("torch version:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
except Exception as e:
    print("Torch check skipped:", e)

## 6) Create persistent run directories

This prepares a local run folder and a matching GCS destination so outputs can be synced off Colab runtime disk.

In [ ]:
from datetime import datetime
from pathlib import Path

run_name = RUN_NAME_OVERRIDE.strip() or datetime.now().strftime("run_%Y%m%d_%H%M%S")
LOCAL_RUN_DIR = Path(f"/content/1307_runs/{run_name}")
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

_output_prefix = GCS_OUTPUT_PREFIX.strip("/")
if not _output_prefix:
    raise ValueError("Set GCS_OUTPUT_PREFIX in the config cell.")

GCS_RUN_URI = f"gs://{GCS_BUCKET}/{_output_prefix}/{run_name}"

print("Run name:", run_name)
print("Local run dir:", LOCAL_RUN_DIR)
print("GCS run uri:", GCS_RUN_URI)

### Before Section 7: prepare local inputs

Run this after Section 6 and before Section 7. It makes the required local files explicit for the selected `ENTRY_SCRIPT`:

- For `create_doe_dataset.py`, it verifies `DOE_GRI_INPUT` exists and tries to sync the parent GCS prefix if it is missing.
- For `doe_geoai.py`, it verifies or extracts `DOE_DATASET_PATH` from `GCS_DATASET_ARCHIVE`.
- For validation runs, set `PREP_MODEL_GCS_URI` and `PREP_LABELBIN_GCS_URI` when `DOE_MODEL_PATH` / `DOE_LABELBIN_PATH` point to local files that need to be copied from GCS.

Leave the optional URI strings blank for normal training, where the model and label bin are written under `LOCAL_RUN_DIR`.

In [ ]:
from pathlib import Path
import shutil

# Optional knobs for this prep cell. Usually leave these as-is.
PREP_FORCE_REFRESH_DATASET = False  # True removes DOE_DATASET_PATH before re-extracting from GCS_DATASET_ARCHIVE.
PREP_MODEL_GCS_URI = ""  # Example: "gs://.../train_brady_19x5d_100ep/doe_geoai_model.h5"
PREP_LABELBIN_GCS_URI = ""  # Example: "gs://.../train_brady_19x5d_100ep/doe_labels.l"


def _gcs_uri(prefix_or_uri):
    prefix_or_uri = str(prefix_or_uri).strip()
    if not prefix_or_uri:
        return ""
    if prefix_or_uri.startswith("gs://"):
        return prefix_or_uri
    return f"gs://{GCS_BUCKET}/{prefix_or_uri.strip('/')}"


def _copy_from_gcs_if_needed(gcs_uri, local_path, label):
    local_path = Path(str(local_path).strip())
    if local_path.exists():
        print(f"{label} exists:", local_path)
        return
    if not gcs_uri.strip():
        print(f"{label} missing and no GCS URI was provided:", local_path)
        return
    local_path.parent.mkdir(parents=True, exist_ok=True)
    run(["gsutil", "cp", _gcs_uri(gcs_uri), str(local_path)])
    print(f"Copied {label}:", local_path)


entry_name = Path(ENTRY_SCRIPT).name

if entry_name == "create_doe_dataset.py":
    gri_path = Path(DOE_GRI_INPUT.strip())
    if not gri_path.exists() and str(gri_path).startswith("/content/"):
        # The notebook mirrors GCS paths under /content, so derive the parent GCS prefix.
        rel_parent = gri_path.parent.relative_to("/content").as_posix()
        source_uri = f"gs://{GCS_BUCKET}/{rel_parent}"
        gri_path.parent.mkdir(parents=True, exist_ok=True)
        print("DOE_GRI_INPUT missing; syncing parent prefix:", source_uri)
        run(["gsutil", "-m", "rsync", "-r", source_uri, str(gri_path.parent)])
    print("DOE_GRI_INPUT exists:", gri_path.exists(), gri_path)

elif entry_name == "doe_geoai.py":
    dataset_path = Path(DOE_DATASET_PATH.strip())
    if PREP_FORCE_REFRESH_DATASET and dataset_path.exists():
        print("Removing existing dataset folder:", dataset_path)
        shutil.rmtree(dataset_path)

    if not dataset_path.exists() and AUTO_DOWNLOAD_DATASET_FROM_GCS:
        archive_prefix = GCS_DATASET_ARCHIVE.strip()
        if not archive_prefix:
            archive_prefix = f"{GCS_DATASET_PREFIX.strip().strip('/')}/{dataset_path.name}.tar.gz"
        archive_uri = _gcs_uri(archive_prefix)
        local_archive = dataset_path.parent / Path(archive_prefix).name
        dataset_path.parent.mkdir(parents=True, exist_ok=True)
        print("Downloading dataset archive:", archive_uri)
        run(["gsutil", "cp", archive_uri, str(local_archive)])
        print("Extracting dataset archive to:", dataset_path.parent)
        run(["tar", "-xzf", str(local_archive), "-C", str(dataset_path.parent)])

    print("DOE_DATASET_PATH exists:", dataset_path.exists(), dataset_path)

    if DOE_MODEL_PATH.strip():
        _copy_from_gcs_if_needed(PREP_MODEL_GCS_URI, DOE_MODEL_PATH, "DOE_MODEL_PATH")
    else:
        print("DOE_MODEL_PATH blank; Section 7 will write model under LOCAL_RUN_DIR for training.")

    if DOE_LABELBIN_PATH.strip():
        _copy_from_gcs_if_needed(PREP_LABELBIN_GCS_URI, DOE_LABELBIN_PATH, "DOE_LABELBIN_PATH")
    else:
        print("DOE_LABELBIN_PATH blank; Section 7 will write labels under LOCAL_RUN_DIR for training.")

else:
    print("No specific prep checks for ENTRY_SCRIPT:", ENTRY_SCRIPT)

## 7) Write run metadata and execute your 1307 entry script

This stores the exact script/args/environment for reproducibility, then runs your job.

For `create_doe_dataset.py`, if `ENTRY_ARGS` is blank the notebook auto-builds args from `DOE_GRI_INPUT`, `DOE_DATASET_OUT_DIR`, and DOE sampling settings.
For `doe_geoai.py`, if `ENTRY_ARGS` is blank the notebook auto-builds required args from `DOE_DATASET_PATH` and output paths. Labels/model default under `LOCAL_RUN_DIR` when overrides are blank.

In [ ]:
import json
import os
import platform
import re
import shlex
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

args = ENTRY_ARGS.strip()

entry = Path(LOCAL_1307_DIR) / ENTRY_SCRIPT
if not entry.exists():
    raise FileNotFoundError(
        f"ENTRY_SCRIPT not found: {entry}\n"
        "Update ENTRY_SCRIPT in the config cell."
    )

# Compatibility patch: synced 1307 tree has `doe_tiff/doe_tiff/*.py` but nested
# `__init__.py` used absolute imports (`from doe_tiff.io ...`) that break when only
# the nested package exists. Fix that, then import the nested package as `dt`.
root1307 = Path(LOCAL_1307_DIR)
nested_init = root1307 / "doe_tiff" / "doe_tiff" / "__init__.py"
if nested_init.is_file():
    init_txt = nested_init.read_text(encoding="utf-8", errors="ignore")
    init_patched = (
        init_txt.replace("from doe_tiff.io import", "from .io import")
        .replace("from doe_tiff.doe_kernel import", "from .doe_kernel import")
    )
    if init_patched != init_txt:
        nested_init.write_text(init_patched, encoding="utf-8")
        print("Patched doe_tiff/doe_tiff/__init__.py with relative imports")

if entry.name == "create_doe_dataset.py":
    src = entry.read_text(encoding="utf-8", errors="ignore")
    orig = src

    bad_block = (
        "try:\n"
        "    from doe_tiff.io import read_gdal_file, frame_image\n"
        "    from doe_tiff.doe_kernel import GeoTiffConvolution\n"
        "except Exception:\n"
        "    from doe_tiff.doe_tiff.io import read_gdal_file, frame_image\n"
        "    from doe_tiff.doe_tiff.doe_kernel import GeoTiffConvolution\n"
    )
    if bad_block in src:
        src = src.replace(bad_block, "")

    src = src.replace("import doe_tiff as dt", "import doe_tiff.doe_tiff as dt")
    src = src.replace("dt.io.read_gdal_file(", "dt.read_gdal_file(")

    # If earlier patches stripped `dt.` from helpers, restore it safely.
    src = re.sub(r"(?<!\.)read_gdal_file\(", "dt.read_gdal_file(", src)
    src = re.sub(r"(?<!\.)frame_image\(", "dt.frame_image(", src)
    src = re.sub(r"(?<!\.)GeoTiffConvolution\(", "dt.GeoTiffConvolution(", src)

    if src != orig:
        entry.write_text(src, encoding="utf-8")
        print("Patched create_doe_dataset.py for nested doe_tiff package layout")

if entry.name == "doe_geoai.py":
    dg = entry.read_text(encoding="utf-8", errors="ignore")
    dg0 = dg
    dg = re.sub(r"(?m)^\s*import\s+keras\s*$", "from tensorflow import keras", dg)
    dg = re.sub(r"(?m)^\s*from\s+keras\.callbacks\s+import\s+", "from tensorflow.keras.callbacks import ", dg)
    dg = re.sub(
        r"(?m)^\s*from\s+keras\.layers\.convolutional\s+import\s+",
        "from tensorflow.keras.layers import ",
        dg,
    )
    dg = re.sub(
        r"(?m)^\s*from\s+keras\.layers\.core\s+import\s+",
        "from tensorflow.keras.layers import ",
        dg,
    )
    dg = re.sub(r"(?m)^\s*from\s+keras\.layers\s+import\s+", "from tensorflow.keras.layers import ", dg)
    dg = re.sub(
        r"(?m)^\s*from\s+keras\.regularizers\s+import\s+",
        "from tensorflow.keras.regularizers import ",
        dg,
    )
    dg = re.sub(r"(?m)^\s*from\s+keras\.models\s+import\s+", "from tensorflow.keras.models import ", dg)
    dg = re.sub(
        r"(?m)^\s*from\s+keras\.optimizers\s+import\s+",
        "from tensorflow.keras.optimizers import ",
        dg,
    )
    dg = re.sub(r"(?m)^\s*from\s+keras\.utils\s+import\s+", "from tensorflow.keras.utils import ", dg)
    dg = re.sub(
        r"(?m)^\s*from\s+keras\s+import\s+backend\s+as\s+K\s*$",
        "from tensorflow.keras import backend as K",
        dg,
    )
    dg = re.sub(r"\bnp\.float\b", "float", dg)
    dg = re.sub(r"\bnp\.int\b", "int", dg)
    dg = re.sub(r"\bnp\.bool\b", "bool", dg)
    if "from tensorflow.keras.utils import multi_gpu_model" in dg and "def multi_gpu_model(model, gpus=None):" not in dg:
        dg = dg.replace(
            "from tensorflow.keras.utils import multi_gpu_model",
            "try:\n    from tensorflow.keras.utils import multi_gpu_model\nexcept Exception:\n    def multi_gpu_model(model, gpus=None):\n        return model",
        )
    dg = re.sub(
        r"ROC_curve_calc\(\s*testY\s*,\s*pre_y2\s*,\s*class_num\s*=\s*8\s*,",
        "ROC_curve_calc( testY, pre_y2, class_num=int(pre_y2_prob.shape[1]),",
        dg,
    )
    if dg != dg0:
        entry.write_text(dg, encoding="utf-8")
        print("Patched doe_geoai.py for tensorflow.keras + ROC class count on Colab")

cmd = [sys.executable, str(entry)]

# Auto-build args when ENTRY_ARGS is left blank.
if not args and entry.name == "create_doe_dataset.py":
    gri = DOE_GRI_INPUT.strip()
    if not gri:
        raise ValueError("Set DOE_GRI_INPUT in the config cell to your .gri path.")
    auto_args = [
        "-i", gri,
        "-c", str(DOE_CHANNELS),
        "-d", DOE_DATASET_OUT_DIR,
        "-s", str(DOE_SAMPLE_COUNT),
        "-k", str(DOE_KERNEL_PIXELS),
    ]
    cmd.extend(auto_args)
elif not args and entry.name == "doe_geoai.py":
    if not DOE_DATASET_PATH.strip():
        raise ValueError(
            "doe_geoai.py needs a training dataset. Set DOE_DATASET_PATH to the extracted tile folder."
        )

    dataset_path = Path(DOE_DATASET_PATH.strip())
    if AUTO_DOWNLOAD_DATASET_FROM_GCS and not dataset_path.exists():
        archive_prefix = GCS_DATASET_ARCHIVE.strip()
        if not archive_prefix:
            archive_prefix = f"{GCS_DATASET_PREFIX.strip().strip('/')}/{dataset_path.name}.tar.gz"
        archive_uri = archive_prefix if archive_prefix.startswith("gs://") else f"gs://{GCS_BUCKET}/{archive_prefix.strip('/')}"
        local_archive = dataset_path.parent / Path(archive_prefix).name
        dataset_path.parent.mkdir(parents=True, exist_ok=True)
        print("Dataset folder missing; downloading:", archive_uri)
        run(["gsutil", "cp", archive_uri, str(local_archive)])
        print("Extracting", local_archive, "to", dataset_path.parent)
        run(["tar", "-xzf", str(local_archive), "-C", str(dataset_path.parent)])

    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Training dataset folder not found after download/extract: {dataset_path}. "
            "Check DOE_DATASET_PATH and GCS_DATASET_ARCHIVE."
        )

    dataset = str(dataset_path)
    labelbin = DOE_LABELBIN_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_labels.l")
    model = DOE_MODEL_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_model.h5")
    plot = DOE_PLOT_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_training_plot.png")
    curves = DOE_CURVES_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_training_curves.csv")

    auto_args = [
        "-d", dataset,
        "-l", labelbin,
        "-m", model,
        "-p", plot,
        "-o", curves,
        "-e", str(DOE_EPOCHS),
        "-b", str(DOE_BATCH_SIZE),
        "-g", str(DOE_GPUS),
        "-k", str(DOE_KERNEL_PIXELS),
        "-c", str(DOE_CHANNELS),
    ]
    extra = DOE_EXTRA_ARGS.strip()
    if extra:
        auto_args.extend(shlex.split(extra))

    cmd.extend(auto_args)
else:
    if args:
        cmd.extend(shlex.split(args))

if AUTO_APPEND_OUTPUT_ARGS:
    cmd.extend([
        OUTPUT_DIR_FLAG,
        str(LOCAL_RUN_DIR),
        SAVE_DIR_FLAG,
        str(LOCAL_RUN_DIR / "checkpoints"),
    ])

# Write a reproducible run manifest before execution.
_git_commit = None
if USE_GIT_FOR_CODE and Path(LOCAL_REPO_DIR, ".git").is_dir():
    try:
        _cp = subprocess.run(
            ["git", "-C", str(Path(LOCAL_REPO_DIR)), "rev-parse", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
        )
        _git_commit = _cp.stdout.strip()
    except Exception as _e:
        _git_commit = f"(error: {_e})"

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "use_git_for_code": USE_GIT_FOR_CODE,
    "git_repo_url": GIT_REPO_URL if USE_GIT_FOR_CODE else None,
    "git_branch": GIT_BRANCH if USE_GIT_FOR_CODE else None,
    "git_commit": _git_commit,
    "local_repo_dir": LOCAL_REPO_DIR if USE_GIT_FOR_CODE else None,
    "entry_script": ENTRY_SCRIPT,
    "entry_args": args,
    "command": cmd,
    "python_executable": sys.executable,
    "python_version": sys.version,
    "platform": platform.platform(),
    "gcp_project": GCP_PROJECT,
    "gcs_bucket": GCS_BUCKET,
    "gcs_code_prefix_1307": GCS_PREFIX_1307,
    "gcs_output_prefix": GCS_OUTPUT_PREFIX,
    "gcs_run_uri": GCS_RUN_URI,
    "sync_dataset_to_gcs": SYNC_DATASET_TO_GCS,
    "gcs_dataset_prefix": GCS_DATASET_PREFIX,
    "local_code_dir": str(LOCAL_1307_DIR),
    "local_run_dir": str(LOCAL_RUN_DIR),
    "auto_append_output_args": AUTO_APPEND_OUTPUT_ARGS,
    "output_dir_flag": OUTPUT_DIR_FLAG,
    "save_dir_flag": SAVE_DIR_FLAG,
    "doe_gri_input": DOE_GRI_INPUT,
    "doe_dataset_out_dir": DOE_DATASET_OUT_DIR,
    "doe_channels": DOE_CHANNELS,
    "doe_sample_count": DOE_SAMPLE_COUNT,
    "doe_kernel_pixels": DOE_KERNEL_PIXELS,
    "doe_dataset_path": DOE_DATASET_PATH,
    "auto_download_dataset_from_gcs": AUTO_DOWNLOAD_DATASET_FROM_GCS,
    "gcs_dataset_archive": GCS_DATASET_ARCHIVE,
    "doe_labelbin_path": DOE_LABELBIN_PATH,
    "doe_model_path": DOE_MODEL_PATH,
    "doe_plot_path": DOE_PLOT_PATH,
    "doe_curves_path": DOE_CURVES_PATH,
    "doe_epochs": DOE_EPOCHS,
    "doe_batch_size": DOE_BATCH_SIZE,
    "doe_gpus": DOE_GPUS,
    "doe_extra_args": DOE_EXTRA_ARGS,
}
manifest_path = Path(LOCAL_RUN_DIR) / "run_config.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote", manifest_path)

run(cmd, cwd=LOCAL_1307_DIR)

if entry.name == "create_doe_dataset.py" and SYNC_DATASET_TO_GCS and (not args):
    out_dir = Path(DOE_DATASET_OUT_DIR).resolve()
    pfx = GCS_DATASET_PREFIX.strip().strip("/")
    if out_dir.is_dir() and pfx:
        archive = out_dir.parent / f"{out_dir.name}.tar.gz"
        if archive.exists():
            archive.unlink()
        try:
            run(["tar", "-czf", str(archive), "-C", str(out_dir.parent), out_dir.name])
            dst = f"gs://{GCS_BUCKET}/{pfx}/{archive.name}"
            run(["gsutil", "-m", "cp", str(archive), dst])
            print("Uploaded tile dataset archive:", dst)
            mpath = Path(LOCAL_RUN_DIR) / "run_config.json"
            if mpath.is_file():
                meta = json.loads(mpath.read_text(encoding="utf-8"))
                meta["dataset_archive_gcs"] = dst
                meta["dataset_archive_local_targz"] = str(archive)
                mpath.write_text(json.dumps(meta, indent=2), encoding="utf-8")
            try:
                os.remove(archive)
                print("Removed local", archive, "(dataset folder still on disk).")
            except OSError:
                pass
        except subprocess.CalledProcessError:
            print("Tile dataset GCS upload failed; data remains on local disk only. See error above.")

## 8) Sync the run folder to GCS

Uploads `/content/1307_runs/...` (manifest, future logs) to `GCS_RUN_URI`.
The **tile dataset** is packed as one **`.tar.gz`** and uploaded with `gsutil cp` (fast vs thousands of `rsync`d files) when `SYNC_DATASET_TO_GCS` is True. To use it in a new session: `gsutil cp` the archive down, then `tar -xzf` into `DOE_DATASET_PATH` (or the path you pass to `doe_geoai.py`).

In [ ]:
from pathlib import Path

if "LOCAL_RUN_DIR" not in globals() or "GCS_RUN_URI" not in globals():
    raise RuntimeError("Run the persistent run directory cell first.")

if not Path(LOCAL_RUN_DIR).exists():
    raise FileNotFoundError(f"Missing local run directory: {LOCAL_RUN_DIR}")

run(["gsutil", "-m", "rsync", "-r", str(LOCAL_RUN_DIR), GCS_RUN_URI])
print("Synced:", GCS_RUN_URI)